# 00 · Setup and data load
**Project:** SQL for demand planning on the Walmart M5 dataset (10 stores, 3,049 items, Jan 2011 – Jun 2016).

**What this notebook does:** loads the raw Kaggle CSVs into DuckDB and builds the base tables every other notebook uses.

| Table | Grain (one row =) | Built from |
|---|---|---|
| `calendar` | day | calendar.csv |
| `prices` | item × store × week | sell_prices.csv |
| `sales_wide` | item × store (one column per day) | sales_train_evaluation.csv |
| `sales_long` | item × store × day | `sales_wide` (UNPIVOT) |
| `sales_daily` | item × store × day, with date, price, revenue | `sales_long` + `calendar` + `prices` |
| `weekly_store_sales` | store × week | `sales_daily` |

**Run order of the notebooks:** 00 → 01 → 02 → 03 → 04 → 05. Notebook 06 holds SQL practice answers.

> **Run one notebook at a time.** DuckDB allows only one open connection to `practice.duckdb`. Before opening another notebook, use *Kernel → Shut Down Kernel* in this one.

In [ ]:
import duckdb

# Connect to the local DuckDB file built in 00_setup_and_load
con = duckdb.connect('../data/practice.duckdb')

# Memory settings: needed for the 59M-row sales_daily table on a laptop
con.sql("SET memory_limit = '4GB'")
con.sql("SET preserve_insertion_order = false")
con.sql("SET temp_directory = '../data/tmp'")

## 1. Load the raw files
DuckDB can read a CSV directly with `SELECT * FROM 'file.csv'`. `CREATE OR REPLACE` lets the cell be re-run safely.

In [ ]:
con.sql("CREATE OR REPLACE TABLE calendar   AS SELECT * FROM '../data/raw/calendar.csv'")
con.sql("CREATE OR REPLACE TABLE prices     AS SELECT * FROM '../data/raw/sell_prices.csv'")
con.sql("CREATE OR REPLACE TABLE sales_wide AS SELECT * FROM '../data/raw/sales_train_evaluation.csv'")

In [ ]:
# Check: one row per item × store → 3,049 items × 10 stores = 30,490
con.sql("SELECT COUNT(*) AS sales_wide_rows FROM sales_wide").df()

,count_star()
0,30490


## 2. Reshape sales from wide to long
M5 stores sales with **one column per day** (`d_1` … `d_1941`). Planning analysis needs **one row per item × store × day**,
so the day columns are turned into rows with `UNPIVOT`.

*IBP comparison: like turning a planning view with days across the columns into a flat key-figure extract.*

In [ ]:
# Every column that starts with "d_" becomes a row: its name goes to column d, its value to units
con.sql("""
CREATE OR REPLACE TABLE sales_long AS
UNPIVOT sales_wide
ON COLUMNS('d_.*')
INTO NAME d VALUE units
""")

In [ ]:
# Check: 30,490 item-stores × 1,941 days = 59,181,090 rows
con.sql("SELECT COUNT(*) AS sales_long_rows FROM sales_long").df()

,count_star()
0,59181090


## 3. Build `sales_daily` (the main fact table)
Adds the real date and week from `calendar`, and the weekly sell price from `prices`.

**Join keys, chosen from each lookup table's grain:**
- `calendar` is one row per day → join on the day key `d`.
- `prices` is one row per item × store × week → join on **all three** keys. Fewer keys would multiply rows.
- `prices` uses a **LEFT JOIN**, so days without a price (item not yet on sale) are kept with a NULL price instead of being dropped.

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE sales_daily AS
SELECT s.item_id,
       s.dept_id,
       s.cat_id,
       s.store_id,
       s.state_id,
       c.date,
       c.wm_yr_wk,                          -- Walmart week, used to join weekly prices
       s.units,
       p.sell_price,                        -- NULL when the item had no price that week
       s.units * p.sell_price AS revenue
FROM sales_long s
INNER JOIN calendar c
        ON s.d = c.d                        -- calendar grain = day
LEFT JOIN prices p
        ON  p.item_id  = s.item_id          -- prices grain = item × store × week,
        AND p.store_id = s.store_id         -- so join on all three keys
        AND p.wm_yr_wk = c.wm_yr_wk
""")

In [ ]:
# Check: same row count as sales_long → the joins did not multiply rows (no fan-out)
con.sql("SELECT COUNT(*) AS sales_daily_rows FROM sales_daily").df()

,count_star()
0,59181090


**Check passed:** 59,181,090 rows, identical to `sales_long`, so every sales row matched exactly one calendar day and at most one price.

## 4. Build `weekly_store_sales` (store × week)
A small summary table used for store-level forecasting and accuracy in notebook 02.

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE weekly_store_sales AS
SELECT store_id,
       wm_yr_wk,
       SUM(units)   AS units,
       SUM(revenue) AS revenue
FROM sales_daily
GROUP BY store_id, wm_yr_wk                 -- output grain: store × week
""")

In [ ]:
# Check: 10 stores × 278 weeks = 2,780 rows
con.sql("""
SELECT COUNT(*)                 AS row_count,
       COUNT(DISTINCT store_id) AS stores,
       COUNT(DISTINCT wm_yr_wk) AS weeks
FROM weekly_store_sales
""").df()

,row_count,stores,weeks
0,2780,10,278


**Check passed:** 2,780 rows = 10 stores × 278 weeks, so the grain is exactly store × week.

> Note: the last week (`wm_yr_wk` 11617) contains only 2 days of data. Later notebooks exclude it from accuracy and feature calculations.